# Notebook 02 — Aerial-Domain Pre-Training of the YOLO26 Detectors (Source Domain)

**Paper artifact:** provenance for the **Aerial** rows of the model-comparison table (`tab:model_metrics_summary`) and the source weights for the Sequential Transfer Learning / negative-transfer analysis.

This notebook trains the **aerial-domain** YOLO26 detectors on **Dataset 1** (*N* = 36,102 aerial corrosion images). These models are the *source* checkpoints from which the Sequential Transfer Learning experiments are initialised, and they also constitute the *Aerial-only* baseline whose sharp collapse on the underwater test set empirically motivates the paper's Direct Transfer design. The pipeline covers: (1) dataset assembly from Roboflow, (2) corrosion-class unification into a single `corrosion` category, and (3) training of the Nano, Medium and Large backbones, exporting the best weights as `modelo-aereo-n.pt`, `modelo-aereo-m.pt` and `modelo-aereo-l.pt`.

**Pipeline stages.**

1. **Library installation and imports** — install and load every required dependency.
2. **Dataset acquisition and preparation** — download the corrosion dataset from Roboflow (5 versions) and merge them into a single training corpus.
3. **Class unification** — rewrite the label files so that every class collapses into a single category: `corrosion`.
4. **Model training** — train three YOLO26 backbones (Nano, Medium and Large) on the prepared aerial dataset.
5. **Model export** — copy the best weights (`best.pt`) of each run into a dedicated folder (`modelos_entrenados`).
6. **Metric visualisation** — display the performance curves and confusion matrices for every trained model.

---

| | |
|---|---|
| **Inputs** | Roboflow project `corrosion-hsmae` (5 versions, *N* = 36,102 images) → `dataset_aereo/data.yaml`; pretrained `yolo26{n,m,l}.pt` weights |
| **Output** | aerial source checkpoints `modelo-aereo-n.pt`, `modelo-aereo-m.pt`, `modelo-aereo-l.pt` in `modelos_entrenados/`; training curves and confusion matrices |
| **Environment** | Ultralytics (YOLO26) · PyTorch (CUDA) · Python 3 · NVIDIA RTX 4090 · AMD Ryzen 5800X |

> **Note on outputs.** All code cells and their outputs are preserved exactly as executed on the original GPU run; the Ultralytics console logs appear in their original form. Re-running under the pinned environment reproduces the same checkpoints.

## 1. Library Installation and Imports

In [1]:
!pip install ultralytics roboflow

import os
import shutil
import glob
import torch
import cv2
import matplotlib.pyplot as plt
from roboflow import Roboflow
from ultralytics import YOLO
from IPython.display import display

## 2. Dataset Acquisition and Preparation

The corrosion dataset is downloaded from Roboflow. It is composed of 5 different versions that are merged into a single dataset for training.

A Roboflow API key is required for the download. It can be found in the official documentation: https://docs.roboflow.com/api-reference/authentication

In [ ]:
import os
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])  # export ROBOFLOW_API_KEY=<your key> (free at app.roboflow.com)
project = rf.workspace("universitas-diponegoro-rfxta").project("corrosion-hsmae")

# Root path for the combined dataset
merged_root = "dataset_corrosion_merged"

# Create the subdirectories for the merged dataset
splits = ["train", "valid", "test"]
for split in splits:
    os.makedirs(os.path.join(merged_root, "images", split), exist_ok=True)
    os.makedirs(os.path.join(merged_root, "labels", split), exist_ok=True)

image_counter = {"train": 0, "valid": 0, "test": 0}

# Iterate over the 5 dataset versions, download and merge them
for version_num in range(1, 6):
    print(f"\nDownloading and processing version {version_num}...")
    version = project.version(version_num)
    dataset = version.download("yolo26")  # Roboflow format identifier (lowercase; see rfapi valid formats)

    for split in splits:
        split_images_path = os.path.join(dataset.location, split, "images")
        split_labels_path = os.path.join(dataset.location, split, "labels")

        if os.path.exists(split_images_path) and os.path.exists(split_labels_path):
            for filename in os.listdir(split_images_path):
                # Copy the image
                src_img = os.path.join(split_images_path, filename)
                dst_img = os.path.join(merged_root, "images", split, f"{image_counter[split]:06d}.jpg")
                shutil.copyfile(src_img, dst_img)

                # Copy the matching label
                label_name = filename.replace(".jpg", ".txt").replace(".png", ".txt")
                src_lbl = os.path.join(split_labels_path, label_name)
                dst_lbl = os.path.join(merged_root, "labels", split, f"{image_counter[split]:06d}.txt")

                if os.path.exists(src_lbl):
                    shutil.copyfile(src_lbl, dst_lbl)

                image_counter[split] += 1

print("\nDataset fully merged!")

# Create the data.yaml file required for YOLO training
yaml_path = os.path.join(merged_root, "data.yaml")
with open(yaml_path, "w") as f:
    f.write(
        "train: images/train\n"
        "val: images/valid\n"
        "test: images/test\n"
        "\n"
        "nc: 1\n"
        "names: ['corrosion']\n"
    )

print(f"data.yaml file created at: {yaml_path}")

## 3. Unification of Corrosion Classes

The original dataset contains multiple classes for corrosion (e.g. `korosi`, `oxido`). To simplify the problem, all of these classes are unified into a single `corrosion` class (index 0).

In [ ]:
def force_first_char_zero(labels_root):
    """
    Walk every .txt file in the (train, valid, test) subdirectories and rewrite the
    first character of each line to '0' so that all classes are unified.
    """
    for split in ("train", "valid", "test"):
        folder = os.path.join(labels_root, split)
        for path in glob.glob(f"{folder}/*.txt"):
            with open(path, "r") as f:
                lines = f.read().splitlines()
            
            new_lines = []
            for line in lines:
                if not line.strip():
                    continue
                new_lines.append("0" + line[1:] + "\n")
            
            with open(path, "w") as f:
                f.writelines(new_lines)

labels_root = "dataset_corrosion_merged/labels"
force_first_char_zero(labels_root)
print("All classes have been unified to '0' in the label files.")

## 4. Model Training

Three models with different architectures are trained to compare their performance:
- **YOLOv26 Nano**: a lightweight, fast model, ideal for real-time applications.
- **YOLOv26 Medium**: a balanced model trading accuracy against computational cost.
- **YOLOv26 Large**: a larger, more accurate model, at the cost of a higher computational load.

Each backbone is pre-trained on the aerial domain (Dataset 1) and its best weights are exported as `modelo-aereo-{n,m,l}.pt`. These checkpoints are the source models reused by the Sequential Transfer Learning experiments in the downstream notebooks.

In [2]:
# --- Auxiliary variables and functions ---

# Path to the dataset configuration file
dataset_path = './dataset_aereo'
yaml_path = f'{dataset_path}/data.yaml'
assert os.path.exists(yaml_path), f"data.yaml not found at {yaml_path}"

# Directory where the trained models are stored
output_models_dir = "modelos_entrenados"
os.makedirs(output_models_dir, exist_ok=True)

# Helper to clear the GPU memory
def clear_gpu_cache():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print("GPU cache cleared.")

# Helper to visualise the training metrics
def show_training_metrics(training_path):
    """Display the results panel and the confusion-matrix images."""
    results_img_path = os.path.join(training_path, 'results.png')
    confusion_matrix_path = os.path.join(training_path, 'confusion_matrix.png')

    if os.path.exists(results_img_path):
        img_results = cv2.imread(results_img_path)
        img_results = cv2.cvtColor(img_results, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(12, 8))
        plt.imshow(img_results)
        plt.title("Training Metrics")
        plt.axis('off')
        plt.show()
    else:
        print(f"Results image not found at {results_img_path}")

    if os.path.exists(confusion_matrix_path):
        img_cm = cv2.imread(confusion_matrix_path)
        img_cm = cv2.cvtColor(img_cm, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(12, 8))
        plt.imshow(img_cm)
        plt.title("Confusion Matrix")
        plt.axis('off')
        plt.show()
    else:
        print(f"Confusion matrix not found at {confusion_matrix_path}")

### 4.1. Training the YOLOv26 Nano Model

In [3]:
import os
import shutil
import torch
from ultralytics import YOLO

def clear_gpu_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        print("✅ GPU cache cleared.")

clear_gpu_cache()

# --- CONFIGURATION ---
yaml_path = './dataset_aereo/data.yaml' 
output_models_dir = './modelos_entrenados'
os.makedirs(output_models_dir, exist_ok=True)

# --- PHASE 1: PRE-TRAINING (SAFE RETRY) ---
print("\n🚀 PHASE 1 RETRY: AERIAL TRAINING (Stable Configuration)")

model_n = YOLO("yolo26n.pt") 

results_n = model_n.train(
    data=yaml_path,
    epochs=50,       
    imgsz=640,
    
    # --- SAFETY ADJUSTMENT ---
    batch=128,       # 128 is the gold standard for the 4090 in YOLO
    workers=8,       # Reduce the load on the system CPU/RAM
    device=0,        
    # ---------------------------
    
    plots=True,
    exist_ok=True,   
    project='runs/detect',
    name='fase1_yolo26n_aereo_v3', # v3 to distinguish runs
    optimizer='auto',
    verbose=True
)

# --- EXPORT ---
best_model_path_n = os.path.join(results_n.save_dir, 'weights', 'best.pt')
final_model_name_n = 'modelo-aereo-n.pt'
destination_path_n = os.path.join(output_models_dir, final_model_name_n)

if os.path.exists(best_model_path_n):
    shutil.copy(best_model_path_n, destination_path_n)
    print(f"\n✅ SUCCESS: Aerial base model saved at: {destination_path_n}")
else:
    print(f"❌ ERROR: the file was not generated.")

✅ GPU cache cleared.

🚀 PHASE 1 RETRY: AERIAL TRAINING (Stable Configuration)
Ultralytics 8.4.5 🚀 Python-3.9.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./dataset_aereo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fase1_yolo26n_aereo_v3, nbs=64, nms=False, op

### Metric Visualisation (YOLOv26 Nano)

The training-results panel and the confusion matrix below summarise how the aerial Nano backbone converged on Dataset 1.

In [4]:
show_training_metrics(results_n.save_dir)

<Figure size 1200x800 with 1 Axes>

<Figure size 1200x800 with 1 Axes>

In [4]:
def clear_gpu_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        print("✅ GPU cache cleared.")

clear_gpu_cache()

✅ GPU cache cleared.


### 4.2. Training the YOLOv26 Medium Model

In [5]:
import os
import shutil
import torch
from ultralytics import YOLO

def clear_gpu_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        print("✅ GPU cache cleared.")

clear_gpu_cache()

# --- CONFIGURATION ---
yaml_path = './dataset_aereo/data.yaml' 
output_models_dir = './modelos_entrenados'
os.makedirs(output_models_dir, exist_ok=True)

# --- PHASE 1: MEDIUM PRE-TRAINING (Aerial Domain) ---
print("\n🚀 STARTING PHASE 1: AERIAL MEDIUM TRAINING")
print("Objective: generate the base weights 'modelo-aereo-m.pt'")

# Load YOLOv26 Medium
model_m = YOLO("yolo26m.pt") 

# Train
results_m = model_m.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    
    # --- RTX 4090 OPTIMISATION (MEDIUM) ---
    batch=32,        # 64 should fit. If it raises OOM, drop to 48 or 32.
    workers=16,      # Use all threads of the Ryzen 5800X
    device=0,
    # --------------------------------------
    
    plots=True,
    exist_ok=True,
    project='runs/detect',
    name='fase1_yolo26m_aereo', # Consistent name
    optimizer='auto',
    verbose=True
)

# --- EXPORT ---
best_model_path_m = os.path.join(results_m.save_dir, 'weights', 'best.pt')
final_model_name_m = 'modelo-aereo-m.pt' # Key name for the next phase
destination_path_m = os.path.join(output_models_dir, final_model_name_m)

if os.path.exists(best_model_path_m):
    shutil.copy(best_model_path_m, destination_path_m)
    print(f"\n✅ SUCCESS: Medium base model saved at: {destination_path_m}")
    print("👉 READY FOR PHASE 2: use this file for the Underwater Transfer Learning.")
else:
    print(f"❌ ERROR: the 'best.pt' file was not generated for the Medium model.")

✅ GPU cache cleared.

🚀 STARTING PHASE 1: AERIAL MEDIUM TRAINING
Objective: generate the base weights 'modelo-aereo-m.pt'
Ultralytics 8.4.5 🚀 Python-3.9.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./dataset_aereo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fa

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



      45/50      18.1G     0.9468     0.4684   0.005464         78        640: 100% ━━━━━━━━━━━━ 1102/1102 3.1it/s 5:58<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 4.1it/s 2.2s0.2s
                   all        569       7828      0.837       0.65      0.727      0.489

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      46/50      18.1G     0.9235     0.4498   0.005298         83        640: 100% ━━━━━━━━━━━━ 1102/1102 3.1it/s 5:58<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 4.1it/s 2.2s0.3s
                   all        569       7828      0.839      0.651      0.729      0.491

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      47/50      18.3G     0.9092     0.4418   0.005177        163        640: 84% ━━━━━━━━━━── 925/1102 3.4it/s 5:00<51.6s

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



### Metric Visualisation (YOLOv26 Medium)

The aerial Medium backbone is the highest-capacity source model that still trains comfortably within the 24 GB RTX 4090 budget; its curves and confusion matrix are shown below.

In [ ]:
show_training_metrics(results_l.save_dir)

### 4.3. Training the YOLOv26 Large Model

In [6]:
import os
import shutil
import torch
from ultralytics import YOLO

def clear_gpu_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        print("✅ GPU cache cleared.")

clear_gpu_cache()

# --- CONFIGURATION ---
yaml_path = './dataset_aereo/data.yaml' 
output_models_dir = './modelos_entrenados'
os.makedirs(output_models_dir, exist_ok=True)

# --- PHASE 1: LARGE PRE-TRAINING (Aerial Domain) ---
print("\n🚀 STARTING PHASE 1: AERIAL LARGE TRAINING")
print("Objective: generate the base weights 'modelo-aereo-l.pt'")

# Load YOLOv26 Large
model_l = YOLO("yolo26l.pt") 

# Train
results_l = model_l.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    
    # --- RTX 4090 OPTIMISATION (LARGE) ---
    batch=16,        # 32 is aggressive but viable in 24GB. If it raises OOM, drop to 16.
    workers=8,       # Keep 8 for system stability
    device=0,
    # -------------------------------------
    
    plots=True,
    exist_ok=True,
    project='runs/detect',
    name='fase1_yolo26l_aereo', 
    optimizer='auto',
    verbose=True
)

# --- EXPORT ---
best_model_path_l = os.path.join(results_l.save_dir, 'weights', 'best.pt')
final_model_name_l = 'modelo-aereo-l.pt' # Input for Phase 2
destination_path_l = os.path.join(output_models_dir, final_model_name_l)

if os.path.exists(best_model_path_l):
    shutil.copy(best_model_path_l, destination_path_l)
    print(f"\n✅ SUCCESS: Large base model saved at: {destination_path_l}")
else:
    print(f"❌ ERROR: the 'best.pt' file was not generated for the Large model.")

✅ GPU cache cleared.

🚀 STARTING PHASE 1: AERIAL LARGE TRAINING
Objective: generate the base weights 'modelo-aereo-l.pt'
Ultralytics 8.4.5 🚀 Python-3.9.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./dataset_aereo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fas

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



       8/50      12.2G      1.723      1.539    0.01126         37        640: 100% ━━━━━━━━━━━━ 2204/2204 4.8it/s 7:43<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 7.4it/s 2.4s0.1s
                   all        569       7828      0.577      0.432      0.443      0.225

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       9/50      12.2G      1.688      1.468    0.01093         75        640: 49% ━━━━━╸────── 1085/2204 5.7it/s 3:47<3:15

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



      14/50      12.1G      1.554      1.201   0.009661        178        640: 100% ━━━━━━━━━━━━ 2204/2204 4.8it/s 7:43<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 7.4it/s 2.4s0.1s
                   all        569       7828      0.723      0.514      0.571      0.326

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      15/50      12.2G      1.538      1.172    0.00945         68        640: 91% ━━━━━━━━━━╸─ 2012/2204 5.5it/s 7:03<35.2s

...

      28/50      12.2G      1.307     0.8571   0.007509        102        640: 100% ━━━━━━━━━━━━ 2204/2204 4.8it/s 7:43<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 7.5it/s 2.4s0.1s
                   all        569       7828      0.803      0.594      0.665       0.42

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      29/50      12.1G      1.284     0.8332   0.007342         74        640: 65% ━━━━━━━╸──── 1430/2204 6.0it/s 4:60<2:10

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



      35/50      12.1G      1.194     0.7311   0.006635         66        640: 100% ━━━━━━━━━━━━ 2204/2204 4.8it/s 7:43<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 7.5it/s 2.4s0.1s
                   all        569       7828      0.829      0.604      0.685      0.443

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      36/50      12.1G      1.179     0.7194   0.006486        106        640: 57% ━━━━━━╸───── 1254/2204 6.0it/s 4:23<2:38

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



      40/50      12.1G      1.118     0.6585   0.006025         78        640: 100% ━━━━━━━━━━━━ 2204/2204 4.8it/s 7:44<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 7.4it/s 2.4s0.1s
                   all        569       7828      0.837      0.615      0.698      0.456
Closing dataloader mosaic
albumentations: __init__() got an unexpected keyword argument 'quality_range'

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      41/50      12.1G      1.068     0.5614   0.006642         39        640: 100% ━━━━━━━━━━━━ 2204/2204 4.8it/s 7:40<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 7.4it/s 2.4s0.1s
                   all        569       7828       0.84      0.619      0.701      0.459

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      42/50      12.1G      1.043     0.5375    

### Metric Visualisation (YOLOv26 Large)

Curves and confusion matrix for the aerial Large backbone, the largest source checkpoint carried forward to the transfer-learning analysis.

In [ ]:
show_training_metrics(results_l.save_dir)